In [ ]:
!pip install requests faiss-cpu

In [ ]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Sample documents (small knowledge base)
documents = [
"Pressure Safety Valve protects equipment from overpressure.",
"A PSV automatically opens when pressure exceeds the design limit.",
"A typical PSV set pressure may be 10 bar in many process vessels.",
"A vessel designed for 10 bar should not exceed 11 bar during relief.",
"Overpressure can occur due to external fire heating of a vessel.",
"A vessel exposed to fire can increase pressure by 5 to 10 bar.",
"Blocked outlet conditions can cause pressure to rise above 15 bar.",
"Thermal expansion of liquid can increase pressure by 2 to 3 bar.",
"A PSV may discharge gas at a rate of 5 kg per second.",
"A large PSV can discharge up to 50 kg per second during emergencies.",
"Flare systems are designed to handle peak loads like 100 kg per second.",
"A flare stack height may be 60 meters in many refineries.",
"Large flare stacks can reach heights of 120 meters.",
"Knockout drums may have a diameter of 3 meters and length of 10 meters.",
"Flare pilots require continuous fuel flow of about 1 kg per hour.",
"Thermal radiation at 4 kilowatt per square meter is safe for long exposure.",
"Thermal radiation at 12.5 kilowatt per square meter can damage equipment.",
"Thermal radiation at 37.5 kilowatt per square meter can damage structures.",
"Radiation above 12.5 kilowatt per square meter may require evacuation.",
"A flare flame length can reach 80 meters during peak flaring.",
"Large flare events may produce flames longer than 100 meters.",
"Consequence modelling predicts hazard zones up to 200 meters.",
"A vapor cloud explosion can produce overpressure of 0.3 bar.",
"Severe explosions may produce overpressure above 1 bar.",
"A catastrophic vessel rupture may release 5000 kg of gas instantly.",
"A pipeline rupture may release gas at 200 kg per second.",
"Emergency blowdown valves may depressurize a vessel within 15 minutes.",
"Blowdown systems can release gas at 30 kg per second.",
"A flare header may operate at pressure around 0.2 bar.",
"Back pressure in flare systems may reach 1 bar during peak flow.",
"Conventional PSVs allow maximum 10 percent back pressure.",
"Balanced bellows PSVs allow up to 50 percent back pressure.",
"Pilot operated PSVs may tolerate up to 100 percent back pressure.",
"Fire case scenarios may involve 3 to 5 vessels relieving simultaneously.",
"A fire case PSV may discharge 10 kg per second from each vessel.",
"If 5 vessels relieve simultaneously the total load may reach 50 kg per second.",
"A large emergency blowdown may add another 20 kg per second load.",
"Total flare load can reach 70 kg per second in severe scenarios.",
"Flare radiation hazard zones may extend up to 150 meters.",
"Personnel safety zones may be set beyond 200 meters.",
"Operators may need to evacuate if radiation exceeds 12.5 kilowatt per square meter.",
"Plant layouts often maintain flare separation distance of 300 meters.",
"Emergency response teams must respond within 5 minutes.",
"PSV inspection is typically conducted every 12 months.",
"Safety audits are often performed every 6 months.",
"Process safety incidents may cost millions of dollars in damage.",
"A major explosion may release energy equivalent to several tons of TNT.",
"Consequence modelling tools calculate thermal radiation contours.",
"Integrated safety systems help reduce risk by more than 80 percent.",
"Proper PSV and flare design significantly reduces catastrophic accident probability."
]

In [ ]:
# Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(documents).toarray()

# Index the document vectors using FAISS (for fast retrieval)
index = faiss.IndexFlatL2(document_vectors.shape[1])
index.add(np.array(document_vectors).astype(np.float32))

In [ ]:
document_vectors

In [ ]:
# Function to retrieve the most relevant document
def retrieve(query, top_n=1):
    query_vector = vectorizer.transform([query]).toarray()
    distances, indices = index.search(query_vector.astype(np.float32), top_n)
    return [documents[i] for i in indices[0]]

In [ ]:
import os
import requests

# Set up Azure OpenAI credentials
api_key = ""
endpoint = "https://hks-testai.services.ai.azure.com/"
deployment_name = "gpt-5"  # Replace with your deployment name

# Create the full endpoint URL
url = f"{endpoint}openai/deployments/{deployment_name}/chat/completions?api-version=2024-10-01-preview"


# Set up headers
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}


In [ ]:
def rag_pipeline(query):
    # Step 1: Retrieve the most relevant document
    retrieved_docs = retrieve(query, top_n=1)
    context = " ".join(retrieved_docs)
    print("Retrieved Document:", context)

    # Step 2: Use Azure OpenAI to generate an answer using the retrieved document
    prompt = f"Based on the following context, answer the question: {query}\n\nContext: {context}"

    # Create the payload
    payload = {
      "messages":[{
          "role":"system",
          "content":"Take content and persona from prompt"
        },{
          "role":"user",
          "content":prompt
      }]
    }



    # Send the request to Azure OpenAI API
    print(f"Sending request to {url}")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        # return result['choices'][0]['text'].strip()
        return result['choices'][0]['message']['content'].strip()
    else:
        return f"Error: {response.status_code}, {response.text}"


In [ ]:
prompt= "“You are a fire and safety engineer responsible for plant emergency response. Using the safety knowledge base, explain what action should be taken if thermal radiation from a flare exceeds 12.5 kW per square meter at a distance of 150 meters.”"

In [ ]:
response = rag_pipeline(prompt)
print("Final Response:", response)
